# Optical tweezers motion analysis

Corrected analysis of the supplied Blender tracking exports. Run all cells from the repository root or `notebooks/`. The ten Brownian-motion tracks are included; the separate trapped-bead recording is not.

This notebook preserves the original cumulative squared-displacement calculation relative to each trajectory’s first position. It is not the usual time-lag-averaged MSD estimator. Tracks are truncated to the shortest length within each bead-size group.

Regression uses a freely fitted intercept consistently in predictions, residuals and plots. The first displacement is at 1/FPS seconds. Nominal OLS errors are descriptive because cumulative observations are correlated. Corrections may change plotted fits and statistics from the historical report. No report PDF or input measurements have been altered.

The speed conversion is distance per frame × FPS × 10⁻⁶ (µm/s to m/s), correcting the old square-root placement. Old saved outputs have been cleared so they cannot be mistaken for corrected results.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import math
import csv
import numpy as np

FPS = 15.0
PIXELS_PER_UM = 11.66
# Works when Jupyter starts in the repository root or notebooks/ directory.
# For another working directory, set DATA_DIR to the cloned repository's data/.
DATA_DIR = next((path.resolve() for path in [Path('data'), Path('../data')]
                 if (path / '1um brownian_bead1.csv').is_file()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Cannot find the supplied data/. Start Jupyter from the repository root or notebooks/.')


In [ ]:
def csv_to_matrix(csv_file):
    """Load semicolon-delimited frame/x/y exports as coordinates in micrometres.

    Reject gaps: the downstream time axis assumes one row per video frame.
    Each trajectory is measured relative to its own first tracked frame.
    """
    with open(csv_file, newline='') as file:
        reader = csv.DictReader(file, delimiter=';')
        rows = [(float(row['frame']), float(row['x']), float(row['y']))
                for row in reader]
    values = np.asarray(rows, dtype=float)
    if values.ndim != 2 or values.shape[0] < 2 or not np.isfinite(values).all():
        raise ValueError('A track needs at least two finite frame/x/y rows.')
    frames = values[:, 0]
    if not np.equal(frames, np.floor(frames)).all() or not np.all(np.diff(frames) == 1):
        raise ValueError('Track frame numbers must be consecutive integers; missing frames need explicit timing.')
    return values[:, 1:3] / PIXELS_PER_UM


In [ ]:
threeum_bead1 = DATA_DIR / '3um brownian ambient light off - use this_bead1.csv'
threeum_bead1_coordinates = csv_to_matrix(threeum_bead1)

threeum_bead2 = DATA_DIR / '3um brownian ambient light off - use this_bead2.csv'
threeum_bead2_coordinates = csv_to_matrix(threeum_bead2)

threeum_bead3 = DATA_DIR / '3um brownian ambient light off - use this_bead3.csv'
threeum_bead3_coordinates = csv_to_matrix(threeum_bead3)

threeum_bead4 = DATA_DIR / '3um brownian ambient light off - use this_bead4.csv'
threeum_bead4_coordinates = csv_to_matrix(threeum_bead4)

threeum_bead5 = DATA_DIR / '3um brownian ambient light off - use this_bead5.csv'
threeum_bead5_coordinates = csv_to_matrix(threeum_bead5)


In [ ]:

x_um = threeum_bead1_coordinates[:, 0]
y_um = threeum_bead1_coordinates[:, 1]

# Plot the points
plt.plot(x_um, y_um, linewidth=0.5)
plt.plot(x_um[0],y_um[0], marker='o', markersize=5, color='green', linestyle='None', label='Initial position')
plt.plot(x_um[-1],y_um[-1], marker='o', markersize=5, color='red', linestyle='None', label='Final position')
plt.xlabel('x-coordinate (${\\mu}$m)')
plt.ylabel('y-coordinate (${\\mu}$m)')
#plt.title('Brownian motion of a 3${\\mu}m$ bead in 0.04% CV')
plt.legend(loc='lower right')
# Show the plot
plt.show()

In [ ]:
def calculate_r_squared(coordinates):
    # Extract the first x and y coordinates as references
    x_reference = coordinates[0, 0]
    y_reference = coordinates[0, 1]

    # Initialize an empty array to store squared distances
    r_squared = []

    # Loop through each row of coordinates starting from the second row
    for i in range(1, coordinates.shape[0]):
        # Calculate squared distance using the formula
        squared_distance = (coordinates[i, 0] - x_reference) ** 2 + (coordinates[i, 1] - y_reference) ** 2
        # Append the squared distance to the array
        r_squared.append(squared_distance)

    # Convert the list to a numpy array
    r_squared = np.array(r_squared)

    return r_squared

In [ ]:
def calculate_time_mean(r_squared):
    time_mean_value = []
    running_sum = 0
    for i, value in enumerate(r_squared):
        running_sum += value
        time_mean_value.append(running_sum / (i + 1))
    return time_mean_value

In [ ]:
def calculate_average_displacement(bead_coordinates_list):
    num_beads = len(bead_coordinates_list)
    min_length = min(len(coords) for coords in bead_coordinates_list)
    time_mean_values_list = []

    # Calculate time_mean_value for each bead up to the length of the shortest list
    for bead_coordinates in bead_coordinates_list:
        r_squared = calculate_r_squared(bead_coordinates[:min_length])
        time_mean_values_list.append(calculate_time_mean(r_squared))

    # Calculate average displacement across all beads up to the length of the shortest list
    average_displacement = [0] * min_length
    for time_mean_values in time_mean_values_list:
        for i, value in enumerate(time_mean_values):
            average_displacement[i] += value

    # Divide the sum by the number of beads
    average_displacement = [value / num_beads for value in average_displacement]

    # Remove the last value from average_displacement before returning

    average_displacement.pop()

    return average_displacement


In [ ]:
threemicronbeads = [threeum_bead1_coordinates, threeum_bead2_coordinates, threeum_bead3_coordinates, threeum_bead4_coordinates, threeum_bead5_coordinates]



In [ ]:
oneum_bead1 = DATA_DIR / '1um brownian_bead1.csv'  
oneum_bead1_coordinates = csv_to_matrix(oneum_bead1)

oneum_bead2 = DATA_DIR / '1um 2 brown_bead1.csv'
oneum_bead2_coordinates = csv_to_matrix(oneum_bead2)

oneum_bead3 = DATA_DIR / '1um 3rd brown_bead1.csv'
oneum_bead3_coordinates = csv_to_matrix(oneum_bead3)

oneum_bead4 = DATA_DIR / '1um 3rd brown_bead2.csv'
oneum_bead4_coordinates = csv_to_matrix(oneum_bead4)

oneum_bead5 = DATA_DIR / '1um 3rd brown_bead3.csv'
oneum_bead5_coordinates = csv_to_matrix(oneum_bead5)

onemicronbeads = [oneum_bead1_coordinates, oneum_bead2_coordinates, oneum_bead3_coordinates, oneum_bead4_coordinates, oneum_bead5_coordinates]

In [ ]:
average_displacement_1um = calculate_average_displacement(onemicronbeads)
average_displacement_3um = calculate_average_displacement(threemicronbeads)

# Create arrays of indices for the x-axis
time_threemicron = np.arange(1, len(average_displacement_3um) + 1) / FPS
time_one_micron = np.arange(1, len(average_displacement_1um) + 1) / FPS

# Perform linear regression
slopeone, interceptone = np.polyfit(time_one_micron, average_displacement_1um, deg=1)
slopethree, interceptthree = np.polyfit(time_threemicron, average_displacement_3um, deg=1)

# Plot the data and linear regression fits
plt.plot(time_threemicron, average_displacement_3um, color='Blue', marker='o', markersize=0.2, markerfacecolor='none', label='3$\\mu$m bead')
plt.plot(time_threemicron, slopethree * time_threemicron + interceptthree, color='Cyan', label='3$\\mu$m Linear Fit')

plt.plot(time_one_micron[:len(time_threemicron)], average_displacement_1um[:len(time_threemicron)], color='Maroon', marker='o', markersize=0.2, markerfacecolor='none', label='1$\\mu$m bead')
plt.plot(time_one_micron, slopeone * time_one_micron + interceptone, color='Red', label='1$\\mu$m Linear Fit')

# Add labels and title
plt.xlabel('Time (seconds)')
plt.ylabel('Average Displacement $<r^2>(t)$ ($\\mu$m$^2$)')
#plt.title('Average Displacement of 3 and 1 $\\mu$m Polystyrene Beads')

# Create separate legends for 1 micron and 3 micron beads and their linear fits
legend1 = plt.legend(handles=[plt.Line2D([0], [0], color='Maroon', label='1$\\mu$m bead'),
                               plt.Line2D([0], [0], color='Red', label='1$\\mu$m Linear Fit')],
                      loc='upper left')

legend2 = plt.legend(handles=[plt.Line2D([0], [0], color='Blue', label='3$\\mu$m bead'),
                               plt.Line2D([0], [0], color='Cyan', label='3$\\mu$m Linear Fit')],
                      loc='lower right')

# Add each legend instance to the plot
plt.gca().add_artist(legend1)
# Show the plot
plt.show()


In [ ]:
x_values = np.arange(1, len(average_displacement_1um) + 1) / FPS

# Perform linear regression
slope, intercept = np.polyfit(x_values, average_displacement_1um, deg=1)

# Calculate predicted values using the linear fit
predicted_values = slope * x_values + intercept

# Calculate the R^2 value
residuals = average_displacement_1um - predicted_values
ss_residual = np.sum(residuals ** 2)
ss_total = np.sum((average_displacement_1um - np.mean(average_displacement_1um)) ** 2)
r_squared = 1 - (ss_residual / ss_total) if ss_total > 0 else np.nan

# Plot average displacement against time
plt.plot(x_values, average_displacement_1um, marker='o', color='Maroon', markersize=0.5, markerfacecolor='none', label='Average Displacement')
plt.xlabel('Time (seconds)')
plt.ylabel('Average Displacement $<r^2>(t)$ ($\\mu$m$^2$)')
plt.title('Average Displacement 1${\\mu}m$ Beads over Time')

# Plot the line of best fit
plt.plot(x_values, predicted_values, 'r-', label=f'Line of Best Fit: y = {slope:.4f} * x + {intercept:.4f}, R² = {r_squared:.2f}')

# Show legend
plt.legend()

# Show plot
plt.grid(True)
plt.show()

In [ ]:
x_values = np.arange(1, len(average_displacement_1um) + 1) / FPS

# Perform linear regression
slope, intercept = np.polyfit(x_values, average_displacement_1um, deg=1)

# Calculate predicted values using the linear fit
predicted_values = slope * x_values + intercept

# Calculate residuals
residuals = average_displacement_1um - predicted_values

# Calculate the sum of squares of residuals
ss_residual = np.sum(residuals ** 2)

# Calculate the standard error of the slope
std_error_slope = np.sqrt(ss_residual / (len(x_values) - 2)) / np.sqrt(np.sum((x_values - np.mean(x_values)) ** 2))

# Descriptive OLS quantity only: cumulative observations are correlated
margin_of_error = 1.96 * std_error_slope  # Nominal normal approximation; correlated observations limit interpretation

# Print standard error of the slope and margin of error
print(f"Descriptive OLS slope standard error: {std_error_slope:.4f}")
print(f"Nominal 1.96 x OLS standard error (not a validated confidence interval): ±{margin_of_error:.4f}")

In [ ]:
x_values = np.arange(1, len(average_displacement_3um) + 1) / FPS

# Perform linear regression
slope, intercept = np.polyfit(x_values, average_displacement_3um, deg=1)

# Calculate predicted values using the linear fit
predicted_values = slope * x_values + intercept

# Calculate the R^2 value
residuals = average_displacement_3um - predicted_values
ss_residual = np.sum(residuals ** 2)
ss_total = np.sum((average_displacement_3um - np.mean(average_displacement_3um)) ** 2)
r_squared = 1 - (ss_residual / ss_total) if ss_total > 0 else np.nan

# Plot average displacement against time
plt.plot(x_values, average_displacement_3um, marker='o', color='Blue', markersize=0.5, markerfacecolor='none', label='Average Displacement')
plt.xlabel('Time (seconds)')
plt.ylabel('Average Displacement $<r^2>(t)$ ($\\mu$m$^2$)')
plt.title('Average Displacement 3${\\mu}m$ Beads over Time')

# Plot the line of best fit
plt.plot(x_values, predicted_values, color='Cyan', label=f'Line of Best Fit: y = {slope:.2f} * x + {intercept:.4f}, R² = {r_squared:.2f}')

# Show legend
plt.legend()

# Show plot
plt.grid(True)
plt.show()

In [ ]:
x_values = np.arange(1, len(average_displacement_3um) + 1) / FPS

# Perform linear regression
slope, intercept = np.polyfit(x_values, average_displacement_3um, deg=1)

# Calculate predicted values using the linear fit
predicted_values = slope * x_values + intercept

# Calculate residuals
residuals = average_displacement_3um - predicted_values

# Calculate the sum of squares of residuals
ss_residual = np.sum(residuals ** 2)

# Calculate the standard error of the slope
std_error_slope = np.sqrt(ss_residual / (len(x_values) - 2)) / np.sqrt(np.sum((x_values - np.mean(x_values)) ** 2))

# Descriptive OLS quantity only: cumulative observations are correlated
margin_of_error = 1.96 * std_error_slope  # Nominal normal approximation; correlated observations limit interpretation

# Print standard error of the slope and margin of error
print(f"Slope: {slope:.4f}")
print(f"Descriptive OLS slope standard error: {std_error_slope:.4f}")
print(f"Nominal 1.96 x OLS standard error (not a validated confidence interval): ±{margin_of_error:.4f}")

## Optional trapped-bead analysis

The historical notebook references `data/laser4_Held_bead.csv`, which is not included in this repository. These cells skip when it is absent. They use the report's historical viscosity value as an explicit input, not a newly validated estimate. The corrected frame-rate conversion changes the speed and force relative to the old code. Do not substitute a Brownian-motion track for the missing trapped-bead recording.


In [ ]:
held_bead = DATA_DIR / 'laser4_Held_bead.csv'
held_bead_coordinates = csv_to_matrix(held_bead) if held_bead.is_file() else None
if held_bead_coordinates is None:
    print('Skipping trapped-bead force analysis: data/laser4_Held_bead.csv is not supplied.')


In [ ]:
def calculate_speed(coordinates):
    """Frame-to-frame speed in m/s for consecutive-frame coordinates in micrometres.

    The first element is zero because no preceding position is available.
    """
    coordinates = np.asarray(coordinates, dtype=float)
    if coordinates.ndim != 2 or coordinates.shape[1] != 2 or len(coordinates) < 2:
        raise ValueError('Expected at least two 2D coordinates.')
    distance_um = np.linalg.norm(np.diff(coordinates, axis=0), axis=1)
    return np.concatenate(([0.0], distance_um * FPS * 1e-6))


In [ ]:
if held_bead_coordinates is not None:
    print(f'Tracked elapsed time: {(len(held_bead_coordinates) - 1) / FPS:.3f} s')


In [ ]:
# No hard-coded duration: elapsed time is derived from the supplied track above.


In [ ]:
def calculate_stokes_force(effective_viscosity, radius_of_bead, max_speed):

    # Calculate the Stokes force
    stokes_force = 6 * math.pi * effective_viscosity * radius_of_bead * max_speed

    return stokes_force

In [ ]:
if held_bead_coordinates is not None:
    held_speed = calculate_speed(held_bead_coordinates)
    max_held_speed = float(np.max(held_speed))
    historical_viscosity_pa_s = 570.38e-6
    bead_radius_m = 1.5e-6
    holding_force_n = calculate_stokes_force(historical_viscosity_pa_s, bead_radius_m, max_held_speed)
    print(f'Stokes-force estimate using historical viscosity: {holding_force_n:.6g} N')
    print('This is a recalculation, not a reproduction or validation of the report result.')


In [ ]:
# A force uncertainty is not reported: timing, localisation, calibration and viscosity uncertainties require propagation.


In [ ]:
if held_bead_coordinates is not None:
    x_um = held_bead_coordinates[:, 0]
    y_um = held_bead_coordinates[:, 1]
    
    # Plot the points
    plt.plot(x_um, y_um, linewidth=0.5)
    plt.plot(x_um[0],y_um[0], marker='o', markersize=5, color='green', linestyle='None', label='Starting position')
    plt.plot(x_um[-1],y_um[-1], marker='o', markersize=5, color='red', linestyle='None', label='Final position')
    plt.xlabel('x-coordinate (${\\mu}$m)')
    plt.ylabel('y-coordinate (${\\mu}$m)')
    #plt.title('Motion of a 3${\\mu}$m bead held by a 658nm laser')
    plt.legend(loc='lower left')
    # Show the plot
    plt.show()
